In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip -q install -U transformers datasets evaluate accelerate scikit-learn scipy pandas peft
!mkdir -p /content/drive/MyDrive/mtl_runs

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 102.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 17.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requir

In [3]:
%%writefile /content/drive/MyDrive/mtl_runs/distilbert_mtl_lora.py
import os
import json
import time
import math
import shutil
import random
from pathlib import Path
from itertools import cycle

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from scipy.stats import pearsonr, spearmanr


# =========================================================
# Config
# =========================================================
MODEL_NAME = "distilbert-base-uncased"
SETTING = "centralized_mtl_lora"
TASKS = ["sst2", "qqp", "stsb"]

BATCH_SIZE = 16
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
MAX_LENGTH = 256
NUM_WORKERS = 2
EARLY_STOPPING_PATIENCE = 3
KEEP_LAST_K_CHECKPOINTS = 2
SEED = 42

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.1
LORA_TARGET_MODULES = ["q_lin", "k_lin", "v_lin", "out_lin"]

BASE_DIR = Path("/content/drive/MyDrive/mtl_runs")
RUN_NAME = "distilbert_mtl_lora_sst2_qqp_stsb"
RUN_DIR = BASE_DIR / RUN_NAME

CHECKPOINT_DIR = RUN_DIR / "checkpoints"
BEST_MODEL_DIR = RUN_DIR / "best_model"
FINAL_MODEL_DIR = RUN_DIR / "final_model"
LOG_CSV_PATH = RUN_DIR / "history.csv"
LOG_JSON_PATH = RUN_DIR / "history.json"
STATE_PATH = RUN_DIR / "training_state.json"


# =========================================================
# Utils
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dirs():
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def save_history(history):
    pd.DataFrame(history).to_csv(LOG_CSV_PATH, index=False)
    save_json(history, LOG_JSON_PATH)


def cleanup_old_checkpoints(keep_k=2):
    ckpts = [p for p in CHECKPOINT_DIR.iterdir() if p.is_dir() and p.name.startswith("epoch_")]
    ckpts = sorted(ckpts, key=lambda x: int(x.name.split("_")[-1]))
    while len(ckpts) > keep_k:
        oldest = ckpts.pop(0)
        shutil.rmtree(oldest, ignore_errors=True)


def get_latest_checkpoint():
    if not CHECKPOINT_DIR.exists():
        return None
    ckpts = [p for p in CHECKPOINT_DIR.iterdir() if p.is_dir() and p.name.startswith("epoch_")]
    if not ckpts:
        return None
    ckpts = sorted(ckpts, key=lambda x: int(x.name.split("_")[-1]))
    return ckpts[-1]


def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable_params, total_params


def safe_float(x):
    if x is None:
        return None
    try:
        if isinstance(x, (float, int, np.floating, np.integer)):
            if np.isnan(x) or np.isinf(x):
                return None
        return float(x)
    except Exception:
        return None


# =========================================================
# Model
# =========================================================
class DistilBertMTL(nn.Module):
    def __init__(self, model_name, use_lora=True):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)

        if use_lora:
            lora_config = LoraConfig(
                task_type=TaskType.FEATURE_EXTRACTION,
                r=LORA_R,
                lora_alpha=LORA_ALPHA,
                lora_dropout=LORA_DROPOUT,
                target_modules=LORA_TARGET_MODULES,
                bias="none",
            )
            self.encoder = get_peft_model(self.encoder, lora_config)

        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(0.1)
        self.sst2_head = nn.Linear(hidden_size, 2)
        self.qqp_head = nn.Linear(hidden_size, 2)
        self.stsb_head = nn.Linear(hidden_size, 1)

    def encode(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state
        cls_rep = last_hidden[:, 0]
        return self.dropout(cls_rep)

    def forward(self, task_name, input_ids, attention_mask, labels=None):
        features = self.encode(input_ids, attention_mask)

        if task_name == "sst2":
            logits = self.sst2_head(features)
            loss = None
            if labels is not None:
                loss = nn.CrossEntropyLoss()(logits, labels.long())
            return {"loss": loss, "logits": logits}

        elif task_name == "qqp":
            logits = self.qqp_head(features)
            loss = None
            if labels is not None:
                loss = nn.CrossEntropyLoss()(logits, labels.long())
            return {"loss": loss, "logits": logits}

        elif task_name == "stsb":
            logits = self.stsb_head(features).squeeze(-1)
            loss = None
            if labels is not None:
                loss = nn.MSELoss()(logits, labels.float())
            return {"loss": loss, "logits": logits}

        else:
            raise ValueError(f"Unsupported task: {task_name}")


def save_model_bundle(model, tokenizer, save_dir, extra_info=None):
    save_dir = Path(save_dir)
    if save_dir.exists():
        shutil.rmtree(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    torch.save(model.state_dict(), save_dir / "pytorch_model.bin")
    tokenizer.save_pretrained(save_dir)

    config_payload = {
        "model_name": MODEL_NAME,
        "setting": SETTING,
        "tasks": TASKS,
        "lora": {
            "enabled": True,
            "r": LORA_R,
            "alpha": LORA_ALPHA,
            "dropout": LORA_DROPOUT,
            "target_modules": LORA_TARGET_MODULES,
        }
    }
    if extra_info is not None:
        config_payload["extra_info"] = extra_info
    save_json(config_payload, save_dir / "mtl_config.json")


# =========================================================
# Data
# =========================================================
def get_task_specs():
    return {
        "sst2": {
            "task_type": "classification",
            "dataset_loader": ("glue", "sst2"),
            "text_cols": ("sentence", None),
            "label_col": "label",
        },
        "qqp": {
            "task_type": "classification",
            "dataset_loader": ("glue", "qqp"),
            "text_cols": ("question1", "question2"),
            "label_col": "label",
        },
        "stsb": {
            "task_type": "regression",
            "dataset_loader": ("glue", "stsb"),
            "text_cols": ("sentence1", "sentence2"),
            "label_col": "label",
        },
    }


def preprocess_dataset(task_name, tokenizer, max_length):
    specs = get_task_specs()[task_name]
    dataset_name, subset_name = specs["dataset_loader"]
    raw = load_dataset(dataset_name, subset_name)

    text_a, text_b = specs["text_cols"]
    label_col = specs["label_col"]
    task_type = specs["task_type"]

    def preprocess_fn(examples):
        if text_b is None:
            enc = tokenizer(
                examples[text_a],
                truncation=True,
                max_length=max_length,
            )
        else:
            enc = tokenizer(
                examples[text_a],
                examples[text_b],
                truncation=True,
                max_length=max_length,
            )

        if task_type == "classification":
            enc["labels"] = examples[label_col]
        else:
            enc["labels"] = [float(x) for x in examples[label_col]]
        return enc

    encoded = raw.map(preprocess_fn, batched=True, remove_columns=raw["train"].column_names)

    if "validation" in encoded:
        val_split = "validation"
    elif "validation_matched" in encoded:
        val_split = "validation_matched"
    else:
        raise ValueError(f"No validation split for task {task_name}")

    return encoded["train"], encoded[val_split]


# =========================================================
# Evaluation
# =========================================================
@torch.no_grad()
def evaluate_task(model, dataloader, device, task_name):
    model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(task_name=task_name, input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs["loss"]
        logits = outputs["logits"]

        total_loss += loss.item()

        if task_name in ["sst2", "qqp"]:
            preds = torch.argmax(logits, dim=-1)
            all_preds.extend(preds.detach().cpu().numpy().tolist())
            all_labels.extend(labels.detach().cpu().numpy().tolist())
        else:
            all_preds.extend(logits.detach().cpu().numpy().tolist())
            all_labels.extend(labels.detach().cpu().numpy().tolist())

    eval_loss = total_loss / max(len(dataloader), 1)

    if task_name in ["sst2", "qqp"]:
        acc = accuracy_score(all_labels, all_preds)
        macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
        precision = precision_score(all_labels, all_preds, average="macro", zero_division=0)
        recall = recall_score(all_labels, all_preds, average="macro", zero_division=0)

        return {
            "eval_loss": eval_loss,
            "accuracy": acc,
            "macro_f1": macro_f1,
            "precision": precision,
            "recall": recall,
            "pearson": None,
            "spearman": None,
        }

    pearson = 0.0
    spearman = 0.0
    if len(set(all_labels)) > 1 and len(set(all_preds)) > 1:
        pearson = float(pearsonr(all_labels, all_preds)[0])
        spearman = float(spearmanr(all_labels, all_preds)[0])

    return {
        "eval_loss": eval_loss,
        "accuracy": None,
        "macro_f1": None,
        "precision": None,
        "recall": None,
        "pearson": pearson,
        "spearman": spearman,
    }


def compute_overall_metric(metrics_by_task):
    sst2_score = metrics_by_task["sst2"]["accuracy"]
    qqp_score = metrics_by_task["qqp"]["accuracy"]
    stsb_score = (metrics_by_task["stsb"]["pearson"] + metrics_by_task["stsb"]["spearman"]) / 2.0
    return float((sst2_score + qqp_score + stsb_score) / 3.0)


# =========================================================
# Checkpoint
# =========================================================
def save_checkpoint(epoch, model, tokenizer, optimizer, scheduler, history, best_metric_so_far, patience_counter):
    ckpt_dir = CHECKPOINT_DIR / f"epoch_{epoch}"
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    torch.save(model.state_dict(), ckpt_dir / "model_state.pt")
    torch.save(
        {
            "epoch": epoch,
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_metric_so_far": best_metric_so_far,
            "patience_counter": patience_counter,
            "history": history,
        },
        ckpt_dir / "trainer_state.pt",
    )
    tokenizer.save_pretrained(ckpt_dir)

    save_json(
        {
            "epoch": epoch,
            "best_metric_so_far": best_metric_so_far,
            "patience_counter": patience_counter,
            "history_length": len(history),
        },
        STATE_PATH,
    )

    cleanup_old_checkpoints(KEEP_LAST_K_CHECKPOINTS)


# =========================================================
# Main
# =========================================================
def main():
    set_seed(SEED)
    ensure_dirs()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("=" * 100)
    print(f"MODEL_NAME: {MODEL_NAME}")
    print(f"SETTING   : {SETTING}")
    print(f"TASKS     : {TASKS}")
    print(f"DEVICE    : {device}")
    print(f"RUN_DIR   : {RUN_DIR}")
    print("Training will continue until early stopping patience=3 is met.")
    print("No fixed max_epochs is used.")
    print("=" * 100)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

    train_datasets = {}
    val_datasets = {}
    train_loaders = {}
    val_loaders = {}

    for task_name in TASKS:
        train_ds, val_ds = preprocess_dataset(task_name, tokenizer, MAX_LENGTH)
        train_datasets[task_name] = train_ds
        val_datasets[task_name] = val_ds

    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    for task_name in TASKS:
        train_loaders[task_name] = DataLoader(
            train_datasets[task_name],
            batch_size=BATCH_SIZE,
            shuffle=True,
            collate_fn=collator,
            num_workers=NUM_WORKERS,
            pin_memory=True,
        )
        val_loaders[task_name] = DataLoader(
            val_datasets[task_name],
            batch_size=BATCH_SIZE,
            shuffle=False,
            collate_fn=collator,
            num_workers=NUM_WORKERS,
            pin_memory=True,
        )

    model = DistilBertMTL(MODEL_NAME, use_lora=True).to(device)

    trainable_params, total_params = count_parameters(model)
    print(f"Trainable params: {trainable_params:,}")
    print(f"Total params    : {total_params:,}")

    for task_name in TASKS:
        print(
            f"{task_name} -> train: {len(train_datasets[task_name])}, "
            f"validation: {len(val_datasets[task_name])}, "
            f"train_batches: {len(train_loaders[task_name])}, "
            f"val_batches: {len(val_loaders[task_name])}"
        )

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    max_train_batches = max(len(train_loaders[t]) for t in TASKS)
    steps_per_epoch = max_train_batches * len(TASKS)
    warmup_steps = max(1, int(steps_per_epoch * WARMUP_RATIO))
    scheduler = get_linear_schedule_with_warmup(
        optimizer=optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=10**12,
    )

    history = []
    start_epoch = 1
    best_metric_so_far = -float("inf")
    patience_counter = 0

    latest_ckpt = get_latest_checkpoint()
    if latest_ckpt is not None:
        print(f"Resuming from checkpoint: {latest_ckpt}")
        model.load_state_dict(torch.load(latest_ckpt / "model_state.pt", map_location=device))

        trainer_state = torch.load(latest_ckpt / "trainer_state.pt", map_location=device)
        optimizer.load_state_dict(trainer_state["optimizer_state_dict"])
        scheduler.load_state_dict(trainer_state["scheduler_state_dict"])
        history = trainer_state.get("history", [])
        best_metric_so_far = trainer_state.get("best_metric_so_far", -float("inf"))
        patience_counter = trainer_state.get("patience_counter", 0)
        start_epoch = trainer_state.get("epoch", 0) + 1

    epoch = start_epoch
    while True:
        model.train()
        epoch_start_time = time.time()

        task_iterators = {task_name: cycle(train_loaders[task_name]) for task_name in TASKS}
        running_task_loss = {task_name: 0.0 for task_name in TASKS}
        running_task_steps = {task_name: 0 for task_name in TASKS}

        print("\n" + "=" * 100)
        print(f"Epoch {epoch} started")
        print("=" * 100)

        for shared_step in range(1, max_train_batches + 1):
            for task_name in TASKS:
                batch = next(task_iterators[task_name])

                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                optimizer.zero_grad()
                outputs = model(task_name=task_name, input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs["loss"]
                loss.backward()
                optimizer.step()
                scheduler.step()

                running_task_loss[task_name] += loss.item()
                running_task_steps[task_name] += 1

            if shared_step % 50 == 0 or shared_step == max_train_batches:
                msg = [f"Epoch {epoch} | Shared Step {shared_step}/{max_train_batches}"]
                for task_name in TASKS:
                    avg_loss = running_task_loss[task_name] / max(running_task_steps[task_name], 1)
                    msg.append(f"{task_name}_loss={avg_loss:.6f}")
                print(" | ".join(msg))

        train_loss_sst2 = running_task_loss["sst2"] / max(running_task_steps["sst2"], 1)
        train_loss_qqp = running_task_loss["qqp"] / max(running_task_steps["qqp"], 1)
        train_loss_stsb = running_task_loss["stsb"] / max(running_task_steps["stsb"], 1)
        train_loss_overall = float((train_loss_sst2 + train_loss_qqp + train_loss_stsb) / 3.0)

        metrics_by_task = {}
        for task_name in TASKS:
            metrics_by_task[task_name] = evaluate_task(model, val_loaders[task_name], device, task_name)

        overall_metric = compute_overall_metric(metrics_by_task)
        is_new_best = overall_metric > best_metric_so_far

        if is_new_best:
            best_metric_so_far = overall_metric
            patience_counter = 0
            save_model_bundle(
                model,
                tokenizer,
                BEST_MODEL_DIR,
                extra_info={
                    "epoch": epoch,
                    "best_metric_so_far": best_metric_so_far,
                },
            )
        else:
            patience_counter += 1

        row = {
            "epoch": epoch,
            "train_loss_overall": safe_float(train_loss_overall),
            "train_loss_sst2": safe_float(train_loss_sst2),
            "train_loss_qqp": safe_float(train_loss_qqp),
            "train_loss_stsb": safe_float(train_loss_stsb),

            "sst2_eval_loss": safe_float(metrics_by_task["sst2"]["eval_loss"]),
            "sst2_accuracy": safe_float(metrics_by_task["sst2"]["accuracy"]),
            "sst2_macro_f1": safe_float(metrics_by_task["sst2"]["macro_f1"]),
            "sst2_precision": safe_float(metrics_by_task["sst2"]["precision"]),
            "sst2_recall": safe_float(metrics_by_task["sst2"]["recall"]),

            "qqp_eval_loss": safe_float(metrics_by_task["qqp"]["eval_loss"]),
            "qqp_accuracy": safe_float(metrics_by_task["qqp"]["accuracy"]),
            "qqp_macro_f1": safe_float(metrics_by_task["qqp"]["macro_f1"]),
            "qqp_precision": safe_float(metrics_by_task["qqp"]["precision"]),
            "qqp_recall": safe_float(metrics_by_task["qqp"]["recall"]),

            "stsb_eval_loss": safe_float(metrics_by_task["stsb"]["eval_loss"]),
            "stsb_pearson": safe_float(metrics_by_task["stsb"]["pearson"]),
            "stsb_spearman": safe_float(metrics_by_task["stsb"]["spearman"]),

            "overall_score": safe_float(overall_metric),
            "time_per_epoch": safe_float(time.time() - epoch_start_time),
            "trainable_params": int(trainable_params),
            "total_params": int(total_params),
            "model_name": MODEL_NAME,
            "dataset_name": "sst2+qqp+stsb",
            "setting": SETTING,
            "best_metric_so_far": safe_float(best_metric_so_far),
            "patience_counter": int(patience_counter),
            "is_new_best": bool(is_new_best),
            "lora_r": LORA_R,
            "lora_alpha": LORA_ALPHA,
            "lora_dropout": LORA_DROPOUT,
        }

        history.append(row)
        save_history(history)

        save_checkpoint(
            epoch=epoch,
            model=model,
            tokenizer=tokenizer,
            optimizer=optimizer,
            scheduler=scheduler,
            history=history,
            best_metric_so_far=best_metric_so_far,
            patience_counter=patience_counter,
        )

        print("\nValidation results:")
        print(json.dumps(row, indent=2, ensure_ascii=False, default=str))

        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print("\nEarly stopping triggered.")
            print(f"Overall validation score did not improve for {EARLY_STOPPING_PATIENCE} consecutive epochs.")
            break

        epoch += 1

    save_model_bundle(
        model,
        tokenizer,
        FINAL_MODEL_DIR,
        extra_info={
            "final_epoch": epoch,
            "best_metric_so_far": best_metric_so_far,
        },
    )
    save_history(history)

    print("\nTraining completed.")
    print(f"History CSV : {LOG_CSV_PATH}")
    print(f"History JSON: {LOG_JSON_PATH}")
    print(f"Best model  : {BEST_MODEL_DIR}")
    print(f"Final model : {FINAL_MODEL_DIR}")
    print(f"Checkpoints : {CHECKPOINT_DIR}")


if __name__ == "__main__":
    main()

Overwriting /content/drive/MyDrive/mtl_runs/distilbert_mtl_lora.py


In [4]:
!python /content/drive/MyDrive/mtl_runs/distilbert_mtl_lora.py

MODEL_NAME: distilbert-base-uncased
SETTING   : centralized_mtl_lora
TASKS     : ['sst2', 'qqp', 'stsb']
DEVICE    : cuda
RUN_DIR   : /content/drive/MyDrive/mtl_runs/distilbert_mtl_lora_sst2_qqp_stsb
Training will continue until early stopping patience=3 is met.
No fixed max_epochs is used.
config.json: 100% 483/483 [00:00<00:00, 1.69MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 215kB/s]
vocab.txt: 232kB [00:00, 10.0MB/s]
tokenizer.json: 466kB [00:00, 9.38MB/s]
README.md: 35.3kB [00:00, 80.9MB/s]
sst2/train-00000-of-00001.parquet: 100% 3.11M/3.11M [00:01<00:00, 1.72MB/s]
sst2/validation-00000-of-00001.parquet: 100% 72.8k/72.8k [00:00<00:00, 296kB/s]
sst2/test-00000-of-00001.parquet: 100% 148k/148k [00:00<00:00, 600kB/s]
Generating train split: 100% 67349/67349 [00:00<00:00, 397668.43 examples/s]
Generating validation split: 100% 872/872 [00:00<00:00, 76961.33 examples/s]
Generating test split: 100% 1821/1821 [00:00<00:00, 96626.32 examples/s]
Map: 100% 67349/67349 [00:09<00